In [2]:
print("Sales columns:")
print(sales.columns.tolist())

print("\nDoes channel exist?")
print("channel" in sales.columns)

Sales columns:
['sales_line_id', 'order_id', 'order_date', 'order_time', 'customer_id', 'product_id', 'store_id', 'salesperson_id', 'promotion_id', 'sales_channel', 'quantity', 'unit_selling_price', 'discount_pct', 'payment_method', 'order_status', 'day_number', 'day_name', 'day_type', 'gross_sales_value']

Does channel exist?
False


In [4]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.13 STEP 1 — DAY-OF-WEEK ANALYSIS
# ============================================================

# Project paths
BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

# ------------------------------------------------------------
# Load all monthly sales files
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(sales_list, ignore_index=True)

print("02.13 STEP 1 — DAY-OF-WEEK ANALYSIS")
print("=" * 70)

print("\nSales files loaded:", len(sales_files))
print("Total sales rows:", len(sales))

# ------------------------------------------------------------
# Convert order date
# ------------------------------------------------------------

sales["order_date"] = pd.to_datetime(sales["order_date"])

# ------------------------------------------------------------
# Create day-of-week fields
# ------------------------------------------------------------

sales["day_number"] = sales["order_date"].dt.dayofweek

sales["day_name"] = sales["order_date"].dt.day_name()

sales["day_type"] = sales["day_number"].apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# ------------------------------------------------------------
# Calculate gross sales
# ------------------------------------------------------------

sales["gross_sales_value"] = (
    sales["quantity"] * sales["unit_selling_price"]
)

# ============================================================
# DAY-OF-WEEK SUMMARY
# ============================================================

dow_summary = (
    sales
    .groupby(
        ["day_number", "day_name", "day_type"],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        orders=("order_id", "nunique"),
        units=("quantity", "sum"),
        gross_sales=("gross_sales_value", "sum")
    )
    .sort_values("day_number")
)

print("\nDay-of-Week Summary:")
print("=" * 70)

print(
    dow_summary.to_string(index=False)
)

# ============================================================
# SALES CHANNEL + WEEKDAY/WEEKEND SUMMARY
# ============================================================

channel_summary = (
    sales
    .groupby(
        ["sales_channel", "day_type"],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        orders=("order_id", "nunique"),
        units=("quantity", "sum"),
        gross_sales=("gross_sales_value", "sum")
    )
)

print("\nSales Channel / Weekday-Weekend Summary:")
print("=" * 70)

print(
    channel_summary.to_string(index=False)
)

# ============================================================
# BASKET BEHAVIOR SUMMARY
# ============================================================

basket_summary = (
    sales
    .groupby(
        ["sales_channel", "day_type"],
        as_index=False
    )
    .agg(
        avg_units_per_line=("quantity", "mean"),
        avg_gross_sales_per_line=("gross_sales_value", "mean")
    )
)

print("\nBasket Behavior Summary:")
print("=" * 70)

print(
    basket_summary.to_string(index=False)
)

# ============================================================
# BASIC VALIDATION
# ============================================================

print("\nValidation:")
print("=" * 70)

print("Day types:", sorted(sales["day_type"].unique()))

print(
    "Sales channels:",
    sorted(sales["sales_channel"].dropna().unique())
)

print(
    "Missing order dates:",
    sales["order_date"].isna().sum()
)

print(
    "Missing day names:",
    sales["day_name"].isna().sum()
)

print(
    "Missing day types:",
    sales["day_type"].isna().sum()
)

print(
    "Missing sales channels:",
    sales["sales_channel"].isna().sum()
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if (
    len(sales) == 2_118_200
    and sales["order_date"].notna().all()
    and sales["day_name"].notna().all()
    and sales["day_type"].notna().all()
    and sales["sales_channel"].notna().all()
    and sales["day_name"].nunique() == 7
):
    print("\n02.13 Step 1 validation: PASSED")
else:
    print("\n02.13 Step 1 validation: FAILED")

02.13 STEP 1 — DAY-OF-WEEK ANALYSIS

Sales files loaded: 32
Total sales rows: 2118200

Day-of-Week Summary:
 day_number  day_name day_type  sales_lines  orders  units  gross_sales
          0    Monday  Weekday       305797  295881 488439 1.502892e+10
          1   Tuesday  Weekday       302458  292694 483838 1.487721e+10
          2 Wednesday  Weekday       302608  292880 484165 1.492190e+10
          3  Thursday  Weekday       299815  290394 479588 1.475279e+10
          4    Friday  Weekday       302406  292818 483968 1.488719e+10
          5  Saturday  Weekend       301905  292182 482599 1.487360e+10
          6    Sunday  Weekend       303211  293284 485178 1.495994e+10

Sales Channel / Weekday-Weekend Summary:
sales_channel day_type  sales_lines  orders   units  gross_sales
       Online  Weekday       378157  363035  604107 1.857636e+10
       Online  Weekend       151418  148936  242486 7.489721e+09
        Store  Weekday      1134927 1007858 1815891 5.589165e+10
        Store 

In [5]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.13 STEP 2 — CREATE WEEKDAY/WEEKEND CONTROL PLAN
# ============================================================

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "weekday_weekend_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Approved weekday/weekend behavior assumptions
# ------------------------------------------------------------

control_plan = pd.DataFrame({
    "day_number": [0, 1, 2, 3, 4, 5, 6],

    "day_name": [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ],

    "day_type": [
        "Weekday",
        "Weekday",
        "Weekday",
        "Weekday",
        "Weekday",
        "Weekend",
        "Weekend"
    ],

    "store_transaction_multiplier": [
        1.00,
        1.00,
        1.00,
        1.00,
        1.20,
        1.30,
        1.35
    ],

    "online_transaction_multiplier": [
        1.00,
        1.00,
        1.00,
        1.00,
        1.10,
        1.15,
        1.20
    ]
})

# ============================================================
# DISPLAY CONTROL PLAN
# ============================================================

print("02.13 STEP 2 — WEEKDAY/WEEKEND CONTROL PLAN")
print("=" * 70)

print(
    control_plan.to_string(index=False)
)

# ============================================================
# VALIDATION
# ============================================================

print("\nValidation")
print("=" * 70)

print("Rows:", len(control_plan))

print(
    "Duplicate day numbers:",
    control_plan["day_number"].duplicated().sum()
)

print(
    "Duplicate day names:",
    control_plan["day_name"].duplicated().sum()
)

print(
    "Missing values:",
    control_plan.isna().sum().sum()
)

print(
    "Invalid store multipliers:",
    (control_plan["store_transaction_multiplier"] <= 0).sum()
)

print(
    "Invalid online multipliers:",
    (control_plan["online_transaction_multiplier"] <= 0).sum()
)

# ============================================================
# WEEKEND VALIDATION
# ============================================================

weekend = control_plan[
    control_plan["day_type"] == "Weekend"
]

weekday = control_plan[
    control_plan["day_type"] == "Weekday"
]

print("\nWeekend store multipliers:")
print(
    weekend[
        ["day_name", "store_transaction_multiplier"]
    ].to_string(index=False)
)

print("\nWeekend online multipliers:")
print(
    weekend[
        ["day_name", "online_transaction_multiplier"]
    ].to_string(index=False)
)

# ============================================================
# SAVE CONTROL PLAN
# ============================================================

CONTROL_PLAN_FILE = (
    OUTPUT_DIR
    / "weekday_weekend_control_plan.csv"
)

control_plan.to_csv(
    CONTROL_PLAN_FILE,
    index=False
)

print("\nControl plan saved to:")
print(CONTROL_PLAN_FILE)

# ============================================================
# FINAL VALIDATION
# ============================================================

if (
    len(control_plan) == 7
    and control_plan["day_number"].nunique() == 7
    and control_plan["day_name"].nunique() == 7
    and control_plan.isna().sum().sum() == 0
    and (control_plan["store_transaction_multiplier"] > 0).all()
    and (control_plan["online_transaction_multiplier"] > 0).all()
    and weekend["store_transaction_multiplier"].max()
        > weekday["store_transaction_multiplier"].max()
    and weekend["online_transaction_multiplier"].max()
        > weekday["online_transaction_multiplier"].max()
):
    print("\n02.13 Step 2 validation: PASSED")
else:
    print("\n02.13 Step 2 validation: FAILED")

02.13 STEP 2 — WEEKDAY/WEEKEND CONTROL PLAN
 day_number  day_name day_type  store_transaction_multiplier  online_transaction_multiplier
          0    Monday  Weekday                          1.00                           1.00
          1   Tuesday  Weekday                          1.00                           1.00
          2 Wednesday  Weekday                          1.00                           1.00
          3  Thursday  Weekday                          1.00                           1.00
          4    Friday  Weekday                          1.20                           1.10
          5  Saturday  Weekend                          1.30                           1.15
          6    Sunday  Weekend                          1.35                           1.20

Validation
Rows: 7
Duplicate day numbers: 0
Duplicate day names: 0
Missing values: 0
Invalid store multipliers: 0
Invalid online multipliers: 0

Weekend store multipliers:
day_name  store_transaction_multiplier
Saturday

In [6]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.13 STEP 3 — APPLY WEEKDAY/WEEKEND BEHAVIOR WEIGHTS
# ============================================================

BASE_DIR = Path.cwd().parent

SALES_DIR = BASE_DIR / "data" / "synthetic"

CONTROL_PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "weekday_weekend_adjusted"
    / "weekday_weekend_control_plan.csv"
)

# ------------------------------------------------------------
# Load control plan
# ------------------------------------------------------------

control_plan = pd.read_csv(CONTROL_PLAN_FILE)

# ------------------------------------------------------------
# Load all sales files
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("02.13 STEP 3 — APPLY WEEKDAY/WEEKEND BEHAVIOR")
print("=" * 70)

print("Sales files loaded:", len(sales_files))
print("Sales rows:", len(sales))

# ------------------------------------------------------------
# Prepare date and day fields
# ------------------------------------------------------------

sales["order_date"] = pd.to_datetime(
    sales["order_date"]
)

sales["day_number"] = (
    sales["order_date"].dt.dayofweek
)

sales["day_name"] = (
    sales["order_date"].dt.day_name()
)

sales["day_type"] = sales["day_number"].apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# ------------------------------------------------------------
# Merge weekday/weekend control plan
# ------------------------------------------------------------

sales = sales.merge(
    control_plan[
        [
            "day_number",
            "store_transaction_multiplier",
            "online_transaction_multiplier"
        ]
    ],
    on="day_number",
    how="left"
)

# ------------------------------------------------------------
# Select multiplier based on sales channel
# ------------------------------------------------------------

sales["day_behavior_multiplier"] = sales[
    "sales_channel"
].astype(str).str.strip().str.lower().map(
    lambda channel:
        sales.loc[
            channel == "store",
            "store_transaction_multiplier"
        ]
        if False else None
)

# Apply channel-specific multiplier safely
sales["day_behavior_multiplier"] = (
    sales["store_transaction_multiplier"]
)

online_mask = (
    sales["sales_channel"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "online"
)

sales.loc[
    online_mask,
    "day_behavior_multiplier"
] = sales.loc[
    online_mask,
    "online_transaction_multiplier"
]

# ------------------------------------------------------------
# Calculate basket metrics
# ------------------------------------------------------------

sales["gross_sales_value"] = (
    sales["quantity"]
    * sales["unit_selling_price"]
)

# ============================================================
# VALIDATION SUMMARY
# ============================================================

summary = (
    sales
    .groupby(
        [
            "day_number",
            "day_name",
            "day_type",
            "sales_channel",
            "day_behavior_multiplier"
        ],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        orders=("order_id", "nunique"),
        units=("quantity", "sum"),
        gross_sales=("gross_sales_value", "sum")
    )
    .sort_values(
        ["day_number", "sales_channel"]
    )
)

print("\nWeekday/Weekend Behavior Summary:")
print("=" * 70)

print(
    summary.to_string(index=False)
)

# ============================================================
# VALIDATION
# ============================================================

print("\nValidation")
print("=" * 70)

print(
    "Missing behavior multipliers:",
    sales["day_behavior_multiplier"].isna().sum()
)

print(
    "Missing day types:",
    sales["day_type"].isna().sum()
)

print(
    "Missing day names:",
    sales["day_name"].isna().sum()
)

print(
    "Rows after mapping:",
    len(sales)
)

print(
    "Unique multipliers:",
    sorted(
        sales["day_behavior_multiplier"]
        .dropna()
        .unique()
    )
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if (
    len(sales) == 2_118_200
    and sales["day_behavior_multiplier"].notna().all()
    and sales["day_type"].notna().all()
    and sales["day_name"].notna().all()
    and (sales["day_behavior_multiplier"] > 0).all()
):
    print("\n02.13 Step 3 validation: PASSED")
else:
    print("\n02.13 Step 3 validation: FAILED")

02.13 STEP 3 — APPLY WEEKDAY/WEEKEND BEHAVIOR
Sales files loaded: 32
Sales rows: 2118200

Weekday/Weekend Behavior Summary:
 day_number  day_name day_type sales_channel  day_behavior_multiplier  sales_lines  orders  units  gross_sales
          0    Monday  Weekday        Online                     1.00        76661   76020 122093 3.739128e+09
          0    Monday  Weekday         Store                     1.00       229136  223510 366346 1.128979e+10
          1   Tuesday  Weekday        Online                     1.00        75764   75104 121077 3.729262e+09
          1   Tuesday  Weekday         Store                     1.00       226694  221270 362761 1.114795e+10
          2 Wednesday  Weekday        Online                     1.00        75532   74944 121037 3.737918e+09
          2 Wednesday  Weekday         Store                     1.00       227076  221576 363128 1.118398e+10
          3  Thursday  Weekday        Online                     1.00        74894   74291 119819 3

In [7]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.13 STEP 4 — VALIDATE WEEKDAY/WEEKEND BEHAVIOR
# ============================================================

BASE_DIR = Path.cwd().parent

SALES_DIR = BASE_DIR / "data" / "synthetic"

CONTROL_PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "weekday_weekend_adjusted"
    / "weekday_weekend_control_plan.csv"
)

# ------------------------------------------------------------
# Load control plan
# ------------------------------------------------------------

control_plan = pd.read_csv(CONTROL_PLAN_FILE)

# ------------------------------------------------------------
# Load sales data
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

# ------------------------------------------------------------
# Prepare day fields
# ------------------------------------------------------------

sales["order_date"] = pd.to_datetime(
    sales["order_date"]
)

sales["day_number"] = (
    sales["order_date"].dt.dayofweek
)

sales["day_name"] = (
    sales["order_date"].dt.day_name()
)

sales["day_type"] = sales["day_number"].apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# ------------------------------------------------------------
# Attach control-plan multipliers
# ------------------------------------------------------------

sales = sales.merge(
    control_plan[
        [
            "day_number",
            "store_transaction_multiplier",
            "online_transaction_multiplier"
        ]
    ],
    on="day_number",
    how="left"
)

# ------------------------------------------------------------
# Determine expected multiplier by channel
# ------------------------------------------------------------

sales["expected_multiplier"] = (
    sales["store_transaction_multiplier"]
)

online_mask = (
    sales["sales_channel"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "online"
)

sales.loc[
    online_mask,
    "expected_multiplier"
] = sales.loc[
    online_mask,
    "online_transaction_multiplier"
]

# ------------------------------------------------------------
# Validate mapped multipliers
# ------------------------------------------------------------

sales["multiplier_valid"] = (
    sales["expected_multiplier"] > 0
)

# ============================================================
# SUMMARY
# ============================================================

validation_summary = (
    sales
    .groupby(
        [
            "day_number",
            "day_name",
            "day_type",
            "sales_channel"
        ],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        average_multiplier=("expected_multiplier", "mean"),
        minimum_multiplier=("expected_multiplier", "min"),
        maximum_multiplier=("expected_multiplier", "max")
    )
    .sort_values(
        ["day_number", "sales_channel"]
    )
)

print("02.13 STEP 4 — WEEKDAY/WEEKEND VALIDATION")
print("=" * 70)

print("\nValidation Summary:")
print(
    validation_summary.to_string(index=False)
)

# ============================================================
# CONTROL-PLAN CHECK
# ============================================================

store_check = (
    control_plan["store_transaction_multiplier"].tolist()
    == sorted(
        control_plan["store_transaction_multiplier"].tolist()
    )
)

online_weekend = control_plan.loc[
    control_plan["day_type"] == "Weekend",
    "online_transaction_multiplier"
]

store_weekend = control_plan.loc[
    control_plan["day_type"] == "Weekend",
    "store_transaction_multiplier"
]

store_weekday = control_plan.loc[
    control_plan["day_type"] == "Weekday",
    "store_transaction_multiplier"
]

online_weekday = control_plan.loc[
    control_plan["day_type"] == "Weekday",
    "online_transaction_multiplier"
]

# Weekend must have stronger maximum activity than weekdays
store_weekend_uplift = (
    store_weekend.max() > store_weekday.max()
)

online_weekend_uplift = (
    online_weekend.max() > online_weekday.max()
)

# Friday should be higher than baseline
friday_store = control_plan.loc[
    control_plan["day_name"] == "Friday",
    "store_transaction_multiplier"
].iloc[0]

friday_online = control_plan.loc[
    control_plan["day_name"] == "Friday",
    "online_transaction_multiplier"
].iloc[0]

friday_store_check = friday_store > 1.0
friday_online_check = friday_online > 1.0

# ============================================================
# FINAL VALIDATION
# ============================================================

print("\nControl-plan checks:")
print(
    "Missing expected multipliers:",
    sales["expected_multiplier"].isna().sum()
)

print(
    "Invalid multipliers:",
    (~sales["multiplier_valid"]).sum()
)

print(
    "Friday Store multiplier:",
    friday_store
)

print(
    "Friday Online multiplier:",
    friday_online
)

print(
    "Store weekend uplift:",
    store_weekend_uplift
)

print(
    "Online weekend uplift:",
    online_weekend_uplift
)

if (
    len(sales) == 2_118_200
    and sales["expected_multiplier"].notna().all()
    and sales["multiplier_valid"].all()
    and friday_store_check
    and friday_online_check
    and store_weekend_uplift
    and online_weekend_uplift
):
    print("\n02.13 Step 4 validation: PASSED")
else:
    print("\n02.13 Step 4 validation: FAILED")

02.13 STEP 4 — WEEKDAY/WEEKEND VALIDATION

Validation Summary:
 day_number  day_name day_type sales_channel  sales_lines  average_multiplier  minimum_multiplier  maximum_multiplier
          0    Monday  Weekday        Online        76661                1.00                1.00                1.00
          0    Monday  Weekday         Store       229136                1.00                1.00                1.00
          1   Tuesday  Weekday        Online        75764                1.00                1.00                1.00
          1   Tuesday  Weekday         Store       226694                1.00                1.00                1.00
          2 Wednesday  Weekday        Online        75532                1.00                1.00                1.00
          2 Wednesday  Weekday         Store       227076                1.00                1.00                1.00
          3  Thursday  Weekday        Online        74894                1.00                1.00              

In [8]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.13 STEP 5 — BASKET BEHAVIOR VALIDATION
# ============================================================

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

# ------------------------------------------------------------
# Load all sales files
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

# ------------------------------------------------------------
# Prepare fields
# ------------------------------------------------------------

sales["order_date"] = pd.to_datetime(
    sales["order_date"]
)

sales["day_number"] = (
    sales["order_date"].dt.dayofweek
)

sales["day_name"] = (
    sales["order_date"].dt.day_name()
)

sales["day_type"] = sales["day_number"].apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# Gross sales
sales["gross_sales_value"] = (
    sales["quantity"]
    * sales["unit_selling_price"]
)

# ============================================================
# BASKET SUMMARY
# ============================================================

basket_summary = (
    sales
    .groupby(
        ["sales_channel", "day_type"],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        orders=("order_id", "nunique"),
        total_units=("quantity", "sum"),
        total_gross_sales=("gross_sales_value", "sum"),
        avg_units_per_line=("quantity", "mean"),
        avg_gross_sales_per_line=("gross_sales_value", "mean")
    )
)

print("02.13 STEP 5 — BASKET BEHAVIOR VALIDATION")
print("=" * 70)

print("\nBasket Summary:")
print(
    basket_summary.to_string(index=False)
)

# ============================================================
# DAY-OF-WEEK BASKET SUMMARY
# ============================================================

daily_basket = (
    sales
    .groupby(
        [
            "day_number",
            "day_name",
            "day_type",
            "sales_channel"
        ],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        avg_units_per_line=("quantity", "mean"),
        avg_gross_sales_per_line=("gross_sales_value", "mean")
    )
    .sort_values(
        ["day_number", "sales_channel"]
    )
)

print("\nDaily Basket Behavior:")
print(
    daily_basket.to_string(index=False)
)

# ============================================================
# WEEKDAY / WEEKEND COMPARISON
# ============================================================

print("\nWeekday vs Weekend Comparison:")
print("=" * 70)

for channel in sorted(
    sales["sales_channel"].dropna().unique()
):

    channel_data = basket_summary[
        basket_summary["sales_channel"] == channel
    ]

    weekday = channel_data[
        channel_data["day_type"] == "Weekday"
    ]

    weekend = channel_data[
        channel_data["day_type"] == "Weekend"
    ]

    if len(weekday) == 1 and len(weekend) == 1:

        weekday_units = weekday[
            "avg_units_per_line"
        ].iloc[0]

        weekend_units = weekend[
            "avg_units_per_line"
        ].iloc[0]

        weekday_value = weekday[
            "avg_gross_sales_per_line"
        ].iloc[0]

        weekend_value = weekend[
            "avg_gross_sales_per_line"
        ].iloc[0]

        units_change = (
            (weekend_units - weekday_units)
            / weekday_units
            * 100
        )

        value_change = (
            (weekend_value - weekday_value)
            / weekday_value
            * 100
        )

        print(f"\nChannel: {channel}")
        print(
            f"Average units/line change: "
            f"{units_change:.2f}%"
        )
        print(
            f"Average gross sales/line change: "
            f"{value_change:.2f}%"
        )

# ============================================================
# BASIC DATA VALIDATION
# ============================================================

print("\nValidation:")
print("=" * 70)

print("Sales rows:", len(sales))
print(
    "Missing quantities:",
    sales["quantity"].isna().sum()
)

print(
    "Missing unit prices:",
    sales["unit_selling_price"].isna().sum()
)

print(
    "Missing sales channels:",
    sales["sales_channel"].isna().sum()
)

print(
    "Missing day types:",
    sales["day_type"].isna().sum()
)

if (
    len(sales) == 2_118_200
    and sales["quantity"].notna().all()
    and sales["unit_selling_price"].notna().all()
    and sales["sales_channel"].notna().all()
    and sales["day_type"].notna().all()
    and len(basket_summary) > 0
):
    print("\n02.13 Step 5 validation: PASSED")
else:
    print("\n02.13 Step 5 validation: FAILED")

02.13 STEP 5 — BASKET BEHAVIOR VALIDATION

Basket Summary:
sales_channel day_type  sales_lines  orders  total_units  total_gross_sales  avg_units_per_line  avg_gross_sales_per_line
       Online  Weekday       378157  363035       604107       1.857636e+10            1.597503              49123.393621
       Online  Weekend       151418  148936       242486       7.489721e+09            1.601434              49463.877762
        Store  Weekday      1134927 1007858      1815891       5.589165e+10            1.600007              49246.913933
        Store  Weekend       453698  432170       725291       2.234382e+10            1.598621              49248.218242

Daily Basket Behavior:
 day_number  day_name day_type sales_channel  sales_lines  avg_units_per_line  avg_gross_sales_per_line
          0    Monday  Weekday        Online        76661            1.592635              48774.834448
          0    Monday  Weekday         Store       229136            1.598815              49271.12

In [9]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.13 STEP 6 — SAVE FINAL WEEKDAY/WEEKEND BEHAVIOR PLAN
# ============================================================

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "weekday_weekend_adjusted"
)

CONTROL_PLAN_FILE = (
    OUTPUT_DIR
    / "weekday_weekend_control_plan.csv"
)

FINAL_FILE = (
    OUTPUT_DIR
    / "weekday_weekend_behavior_plan.csv"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load validated control plan
# ------------------------------------------------------------

control_plan = pd.read_csv(CONTROL_PLAN_FILE)

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "day_number",
    "day_name",
    "day_type",
    "store_transaction_multiplier",
    "online_transaction_multiplier"
]

missing_columns = [
    column
    for column in required_columns
    if column not in control_plan.columns
]

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

duplicate_days = control_plan["day_number"].duplicated().sum()

missing_values = (
    control_plan[required_columns]
    .isna()
    .sum()
    .sum()
)

invalid_store_multipliers = (
    control_plan["store_transaction_multiplier"] <= 0
).sum()

invalid_online_multipliers = (
    control_plan["online_transaction_multiplier"] <= 0
).sum()

weekday_count = (
    control_plan["day_type"] == "Weekday"
).sum()

weekend_count = (
    control_plan["day_type"] == "Weekend"
).sum()

# ------------------------------------------------------------
# Display final plan
# ------------------------------------------------------------

print("02.13 STEP 6 — FINAL WEEKDAY/WEEKEND BEHAVIOR PLAN")
print("=" * 70)

print("\nFinal Plan:")
print(
    control_plan[required_columns]
    .to_string(index=False)
)

print("\nValidation")
print("=" * 70)

print("Rows:", len(control_plan))
print("Missing required columns:", len(missing_columns))
print("Duplicate days:", duplicate_days)
print("Missing values:", missing_values)
print("Invalid store multipliers:", invalid_store_multipliers)
print("Invalid online multipliers:", invalid_online_multipliers)
print("Weekdays:", weekday_count)
print("Weekends:", weekend_count)

# ------------------------------------------------------------
# Save final behavior plan
# ------------------------------------------------------------

control_plan[
    required_columns
].to_csv(
    FINAL_FILE,
    index=False
)

print("\nFinal behavior plan saved to:")
print(FINAL_FILE)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if (
    len(control_plan) == 7
    and len(missing_columns) == 0
    and duplicate_days == 0
    and missing_values == 0
    and invalid_store_multipliers == 0
    and invalid_online_multipliers == 0
    and weekday_count == 5
    and weekend_count == 2
):
    print("\n02.13 Step 6 validation: PASSED")
else:
    print("\n02.13 Step 6 validation: FAILED")

02.13 STEP 6 — FINAL WEEKDAY/WEEKEND BEHAVIOR PLAN

Final Plan:
 day_number  day_name day_type  store_transaction_multiplier  online_transaction_multiplier
          0    Monday  Weekday                          1.00                           1.00
          1   Tuesday  Weekday                          1.00                           1.00
          2 Wednesday  Weekday                          1.00                           1.00
          3  Thursday  Weekday                          1.00                           1.00
          4    Friday  Weekday                          1.20                           1.10
          5  Saturday  Weekend                          1.30                           1.15
          6    Sunday  Weekend                          1.35                           1.20

Validation
Rows: 7
Missing required columns: 0
Duplicate days: 0
Missing values: 0
Invalid store multipliers: 0
Invalid online multipliers: 0
Weekdays: 5
Weekends: 2

Final behavior plan saved to:
c: